# Schwab Trader — Google Colab Quickstart

Run the cells **top to bottom**. You'll log into Schwab once in your phone's browser, then this notebook can read balances/positions and place orders.

**Order of operations:**
1. Install + enter credentials
2. Get the login URL → approve in your browser
3. Paste the redirect URL back → saves `token.json`
4. View balances & positions
5. Place a limit order

> ⚠️ This trades a **real** account. Test with a limit price far from the market so nothing fills.

> 🔒 This notebook never stores your secret in the file — you paste it at runtime (hidden). Safe to keep in GitHub.

## 1. Install the library

In [ ]:
!pip install -q schwab-trader python-dotenv websockets
print('Installed.')

## 2. Enter your credentials

Running this cell will prompt you to paste your App Key and Secret (input is hidden). They live only in memory for this session.

Tip: you can instead store them in Colab **Secrets** (the 🔑 icon) as `SCHWAB_APP_KEY` and `SCHWAB_APP_SECRET` and this cell will pick them up automatically.

In [ ]:
import getpass

def _secret(name, prompt):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            print(f'Loaded {name} from Colab Secrets.')
            return val
    except Exception:
        pass
    return getpass.getpass(prompt)

APP_KEY      = _secret('SCHWAB_APP_KEY', 'Paste your App Key: ')
APP_SECRET   = _secret('SCHWAB_APP_SECRET', 'Paste your App Secret: ')
CALLBACK_URL = 'https://127.0.0.1'   # must EXACTLY match your app's registered callback
TOKEN_PATH   = 'token.json'

print('Credentials loaded. App Key starts with:', APP_KEY[:4] + '...')

## 3. Get the login URL

Run this, then **tap the printed URL**, log into Schwab, and approve the app.

In [ ]:
import json, os
from datetime import datetime, timezone
from schwab import SchwabAuth, SchwabClient

auth = SchwabAuth(client_id=APP_KEY, client_secret=APP_SECRET, redirect_uri=CALLBACK_URL)

def save_tokens(auth, path=TOKEN_PATH):
    data = {
        'access_token': auth.access_token,
        'refresh_token': auth.refresh_token,
        'token_expiry': auth.token_expiry.isoformat() if auth.token_expiry else None,
        'saved_at': datetime.now(timezone.utc).isoformat(),
    }
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    try:
        os.chmod(path, 0o600)
    except OSError:
        pass

def load_tokens(auth, path=TOKEN_PATH):
    if not os.path.exists(path):
        return False
    with open(path) as f:
        d = json.load(f)
    auth.access_token = d.get('access_token')
    auth.refresh_token = d.get('refresh_token')
    if d.get('token_expiry'):
        exp = datetime.fromisoformat(d['token_expiry'])
        if exp.tzinfo is None:
            exp = exp.replace(tzinfo=timezone.utc)
        auth.token_expiry = exp
    return auth.access_token is not None

# Persist tokens automatically whenever they're created or refreshed.
_orig_update = auth._update_tokens
def _update_and_save(td):
    _orig_update(td)
    try:
        save_tokens(auth)
    except Exception:
        pass
auth._update_tokens = _update_and_save

print('1) Tap this URL, log in to Schwab, and approve:\n')
print('   ' + auth.get_authorization_url())
print('\n2) Your browser redirects to https://127.0.0.1/?code=...  (the page will NOT load — that is normal).')
print('3) Copy the ENTIRE address-bar URL, then run the next cell and paste it.')

## 4. Paste the redirect URL → save tokens

The `code` is only valid for ~30 seconds, so paste promptly. If it expires, re-run cell 3.

In [ ]:
from urllib.parse import urlparse, parse_qs, unquote

pasted = input('Paste the full redirected URL (or just the code): ').strip()
if 'code=' in pasted:
    code = parse_qs(urlparse(pasted).query)['code'][0]
else:
    code = unquote(pasted)

auth.exchange_code_for_tokens(code)
save_tokens(auth)
print('✅ Tokens saved to', TOKEN_PATH)
print('   access token expires:', auth.token_expiry)

## 5. Build the client + view balances & positions

Read-only — no risk.

In [ ]:
client = SchwabClient(client_id=APP_KEY, client_secret=APP_SECRET,
                      redirect_uri=CALLBACK_URL, auth=auth)

def as_dict(o):
    if o is None: return {}
    if isinstance(o, dict): return o
    if hasattr(o, 'model_dump'): return o.model_dump(by_alias=True)
    if hasattr(o, 'root'): return as_dict(o.root)
    return dict(o)

def money(v):
    try: return f'${float(v):,.2f}'
    except Exception: return str(v)

accounts = client.get_accounts(include_positions=True)
print(f'Found {len(accounts)} account(s).')
for acct in accounts:
    sec = as_dict(getattr(acct, 'securities_account', acct))
    if 'securitiesAccount' in sec:
        sec = as_dict(sec['securitiesAccount'])
    print('\n' + '='*60)
    print(f"Account {sec.get('accountNumber','?')}  ({sec.get('type','')})")
    print('='*60)
    bals = as_dict(sec.get('currentBalances'))
    for k in ['liquidationValue','cashBalance','availableFunds','buyingPower','equity']:
        if k in bals:
            print(f'  {k:<18} {money(bals[k])}')
    positions = [as_dict(p) for p in (sec.get('positions') or [])]
    print(f'\n  Positions ({len(positions)}):')
    for p in positions:
        sym = as_dict(p.get('instrument')).get('symbol','?')
        qty = (p.get('longQuantity') or 0) - (p.get('shortQuantity') or 0)
        print(f"    {sym:<8} qty={qty:<8.4g} avg={money(p.get('averagePrice',0))} mkt={money(p.get('marketValue',0))}")

## 6. Place a limit order

Edit the four variables below. A SELL at a very high price (or BUY at a very low price) will be **accepted but won't fill** — ideal for a safe first test. You'll be asked to confirm before it sends.

In [ ]:
from schwab.models.generated.trading_models import Instruction

SIDE   = 'SELL'      # 'BUY' or 'SELL'
SYMBOL = 'AAPL'
QTY    = 1
LIMIT  = 9999.00     # SELL high so it won't fill (safe test)

nums = client.get_account_numbers().accounts
acct = nums[0]   # first account; change index if you have several
print(f'Account: {acct.account_number}')
print(f'Order:   {SIDE} {QTY} {SYMBOL} LIMIT @ ${LIMIT:,.2f} (DAY)')

if input("Type 'yes' to send: ").strip().lower() == 'yes':
    instr = Instruction.buy if SIDE == 'BUY' else Instruction.sell
    order = client.create_limit_order(symbol=SYMBOL, quantity=QTY,
                                      limit_price=LIMIT, instruction=instr)
    client.place_order(acct.hash_value, order)
    print('✅ Order submitted. Check the Schwab app for status; cancel it there after testing.')
else:
    print('Cancelled — no order sent.')

## (Later) Reconnect without logging in again

Your access token lasts ~30 min and refreshes automatically; the refresh token lasts ~7 days. As long as `token.json` still exists, run the cell below to rebuild the client — no browser needed. After ~7 days, re-run cells 3–4 to log in again.

> Colab wipes files when the runtime resets, so `token.json` won't survive forever. To keep it across sessions, mount Google Drive and set `TOKEN_PATH` to a Drive path.

In [ ]:
auth = SchwabAuth(client_id=APP_KEY, client_secret=APP_SECRET, redirect_uri=CALLBACK_URL)
_orig_update = auth._update_tokens
auth._update_tokens = lambda td: (_orig_update(td), save_tokens(auth))

if not load_tokens(auth):
    print('No token.json found — run cells 3–4 to log in.')
else:
    auth.ensure_valid_token()   # refreshes if needed
    client = SchwabClient(client_id=APP_KEY, client_secret=APP_SECRET,
                          redirect_uri=CALLBACK_URL, auth=auth)
    print('✅ Reconnected. Client ready.')